In [1]:
import os
import yaml
from ultralytics import YOLO, settings
import torch

settings.update({'datasets_dir': r'E:\Projects\CCTV_horkang'})
print("Settings updated to project directory.")

Settings updated to project directory.


In [2]:
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    # Verify it sees the 5060 and the 12.0 capability
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    # Critical test: perform a calculation on the GPU
    # If this doesn't crash, your 5060 is ready!
    test_tensor = torch.randn(1024, 1024).cuda()
    result = test_tensor @ test_tensor
    print("✅ GPU calculation successful!")

CUDA available: True
GPU: NVIDIA GeForce RTX 5060
Compute Capability: (12, 0)
✅ GPU calculation successful!


In [3]:
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if device=='0' else 'CPU'})")

Using device: 0 (NVIDIA GeForce RTX 5060)


In [4]:
project_root = r'E:\Projects\CCTV_horkang\train_dataset_v26'
train_images = os.path.join(project_root, 'train', 'images')
val_images = os.path.join(project_root, 'valid', 'images')
test_images = os.path.join(project_root, 'test', 'images')

data_config = {
    'train': train_images,
    'val': val_images,
    'test': test_images,
    'nc': 1,
    'names': ['human']
}

yaml_path = os.path.join(project_root, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)


In [ ]:
model = YOLO('yolo26m.pt')
model.train(
    data=r'E:\Projects\CCTV_horkang\train_dataset_v26\data.yaml',
    epochs=100,      # High epochs for better accuracy
    imgsz=960,       # Standard resolution
    batch=8,        # Adjust to 8 or 4 if you get "Out of Memory"
    device=device,   # Uses your NVIDIA GPU
    workers=4,       # Number of CPU threads loading data
    name='human_detector_v3',
    exist_ok=True,
    patience=20       # Early stopping patience
)

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260320+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Projects\CCTV_horkang\train_dataset_v26\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosa

In [ ]:
best_model = YOLO('runs/detect/human_detector_v2/weights/best.pt')
metrics = best_model.val(
    data=r'E:\Projects\CCTV_horkang\train_dataset_v26\data.yaml', 
    split='test',
    workers=0,
    device=0
)
print("--- Test Set Results ---")
print(f"mAP50-95: {metrics.box.map:.3f}") # Overall accuracy
print(f"mAP50:    {metrics.box.map50:.3f}") # Accuracy at 50% overlap (Standard)
print(f"Precision: {metrics.box.mp:.3f}")   # How many detections were correct
print(f"Recall:    {metrics.box.mr:.3f}")

Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260320+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.20.1 ms, read: 7.02.2 MB/s, size: 38.3 KB)
val: Scanning E:\Projects\CCTV_horkang\train_dataset_v26\test\labels... 1149 images, 3 backgrounds, 15 corrupt: 100% ━━━━━━━━━━━━ 1149/1149 683.9it/s 1.7s0.0s
val: E:\Projects\CCTV_horkang\train_dataset_v26\test\images\train_all_data--105-_jpg.rf.999dd4547d795e9e737731bbeaf04b9f.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: E:\Projects\CCTV_horkang\train_dataset_v26\test\images\train_all_data--108-_jpg.rf.d92792db36c637eabbeb6112947b7e4b.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: E:\Projects\CCTV_horkang\train_dataset_v26\test\images\train_all_data--119-_jpg.rf.1a044502c76e58545394b6b1